In [2]:
!pip install -q pypdf faiss-cpu numpy google-generativeai

In [1]:
import numpy as np
from pypdf import PdfReader
import google.generativeai as genai
import faiss 
from dotenv import load_dotenv 
import os

/tmp/ipykernel_37235/1122640401.py:3: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  import google.generativeai as genai


In [2]:
load_dotenv()

True

In [3]:
GEMINI_KEY=os.getenv('GEMINI_KEY')

In [4]:
genai.configure(api_key=GEMINI_KEY)

In [5]:
model = genai.GenerativeModel("gemini-2.5-flash")

In [6]:
pdf_path = "python_tutorial.pdf"

pdf_reader = PdfReader(pdf_path)

text = ""
for page in pdf_reader.pages:
    text += page.extract_text()

print("PDF loaded")

PDF loaded


In [7]:
def chunk_text(text, size=800, overlap=150):
    chunks = []
    start = 0

    while start < len(text):
        end = start + size
        chunks.append(text[start:end])
        start = end - overlap

    return chunks

chunks = chunk_text(text)

print("Chunks:", len(chunks))

Chunks: 119


In [8]:
def do_embed(text):
    vec = np.zeros(384)

    for i, c in enumerate(text[:384]):
        vec[i % 384] += ord(c)

    return vec / (np.linalg.norm(vec) + 1e-9)

embeddings = np.array([do_embed(c) for c in chunks]).astype("float32")

In [9]:
index = faiss.IndexFlatL2(384)
index.add(embeddings)

print("FAISS Ready")

FAISS Ready


In [10]:
def retrieve(query, k=3):
    q_vec = do_embed(query).astype("float32").reshape(1, -1)

    _, idx = index.search(q_vec, k)

    return [chunks[i] for i in idx[0]]

In [11]:
def ask_gemini(question, context):
    prompt = f"""
You are an AI assistant.

Read and digest the qustions and Answer ONLY using the context below.

Context:
{context}

Question:
{question}

Answer:
"""

    response = model.generate_content(prompt)
    return response.text

In [12]:
while True:
    q = input("\nAsk a question (type 'exit'): ")

    if q.lower() == "exit":
        break

    docs = retrieve(q)

    context = "\n".join(docs)

    answer = ask_gemini(q, context)

    print("\nAnswer:\n", answer)


Answer:
 The context provides information about various topics related to Python, such as "Wrapper Classes", "Enums", "Reflection", "Errors & Exceptions", "Database Access", "Weak References", "Serialization", "Templating", "Output Formatting", "Decorators", "Recursion", "Regular Expressions", and "PIP". However, it does not define what Python is.

Answer:
 No question was provided.
